# 🏋️ CUDA Lab Exercises

Work through these exercises in order. Each kernel has **`# TODO` markers** where you fill in the CUDA code.  
Run the test cell after each exercise to check your answer.

| Exercise | Topic | Difficulty |
|----------|-------|------------|
| 1 | In-place vector scale | ⭐ Warm-up |
| 2 | Parallel max reduction | ⭐⭐ |
| 3 | ReLU activation | ⭐⭐ |
| 4 | Matrix transpose — naive | ⭐⭐⭐ |
| 5 | Matrix transpose — tiled (bank-conflict free) | ⭐⭐⭐⭐ |

> **Tip:** All kernels use PyCUDA `SourceModule` — write raw CUDA C inside the triple-quoted string.

In [ ]:
import numpy as np
import pycuda.autoinit
import pycuda.driver as cuda
from pycuda.compiler import SourceModule
import pycuda.gpuarray as gpuarray

def benchmark_gpu(func, warmup=5, repeats=50):
    start, end = cuda.Event(), cuda.Event()
    for _ in range(warmup): func()
    cuda.Context.synchronize()
    start.record()
    for _ in range(repeats): func()
    end.record(); end.synchronize()
    return start.time_till(end) * 1e-3 / repeats

print(f"GPU: {cuda.Device(0).name()}  ✓")

---
## Exercise 1 — In-Place Vector Scale  ⭐

**Task:** Write a kernel that multiplies every element of array `a` by scalar `s` **in place**.  
`a[i] = a[i] * s`

**Key concept:** Thread index calculation — `blockIdx.x * blockDim.x + threadIdx.x`

```
Before: [1, 2, 3, 4, 5]
s = 3
After:  [3, 6, 9, 12, 15]
```

In [ ]:
scale_src = """
__global__ void vector_scale(float *a, float s, int n) {
    // TODO: compute this thread's global index
    int idx = ???;

    // TODO: bounds check, then scale a[idx] by s
    ???
}
"""

mod_scale  = SourceModule(scale_src)
vector_scale = mod_scale.get_function("vector_scale")

In [ ]:
# ── Test Ex 1 ─────────────────────────────────────────────────────────────────
N  = 1 << 20
h  = np.random.rand(N).astype(np.float32)
d  = gpuarray.to_gpu(h)
s  = np.float32(3.14)

BLOCK = 256
GRID  = int(np.ceil(N / BLOCK))
vector_scale(d, s, np.int32(N), block=(BLOCK,1,1), grid=(GRID,1,1))
cuda.Context.synchronize()

expected = h * s
err = np.max(np.abs(d.get() - expected))
print(f"Max error : {err:.2e}")
print("✓ PASSED" if err < 1e-5 else "✗ FAILED — check your TODO")

t = benchmark_gpu(lambda: vector_scale(d, s, np.int32(N), block=(BLOCK,1,1), grid=(GRID,1,1)))
print(f"Time: {t*1e3:.3f} ms  |  BW: {N*4/t/1e9:.1f} GB/s")

In [ ]:
# ── Solution Ex 1 (try yourself first!) ──────────────────────────────────────
scale_solution = """
__global__ void vector_scale(float *a, float s, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;  // global thread ID
    if (idx < n) {
        a[idx] = a[idx] * s;
    }
}
"""
# Uncomment to check:
# mod_sol1 = SourceModule(scale_solution)
# vector_scale = mod_sol1.get_function("vector_scale")

---
## Exercise 2 — Parallel Max Reduction  ⭐⭐

**Task:** Find the maximum element in each block using shared-memory tree reduction.  
Thread 0 of each block writes the block's max to `g_odata[blockIdx.x]`.

**Hint:** This is almost identical to the sum-reduction tree from Section 1 of the tutorial —  
the only change is replacing `+` with `max()`.

```
Input: [3, 1, 4, 1, 5, 9, 2, 6]
Max:   9

Tree step 1: max(3,6) max(1,2) max(4,9) max(1,5) → [6, 2, 9, 5]
Tree step 2: max(6,9) max(2,5)                   → [9, 5]
Tree step 3: max(9,5)                             → [9]
```

In [ ]:
max_reduce_src = """
__global__ void max_reduction(const float *g_idata, float *g_odata, int n) {
    extern __shared__ float sdata[];

    unsigned int tid = threadIdx.x;
    unsigned int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // TODO: Load one element (or -infinity if out of bounds) into shared memory
    sdata[tid] = ???;
    __syncthreads();

    // TODO: Tree reduction — but use fmaxf() instead of +
    for (unsigned int s = blockDim.x / 2; s > 0; s >>= 1) {
        if (tid < s) {
            sdata[tid] = ???;
        }
        __syncthreads();
    }

    // TODO: Thread 0 writes the block's max to global output
    if (tid == 0) ???;
}
"""

mod_max = SourceModule(max_reduce_src)
max_reduction_gpu = mod_max.get_function("max_reduction")

In [ ]:
# ── Test Ex 2 ─────────────────────────────────────────────────────────────────
N2   = 1 << 20
h2   = np.random.rand(N2).astype(np.float32)
d2   = gpuarray.to_gpu(h2)

BLOCK = 256
GRID  = int(np.ceil(N2 / BLOCK))
d_partial = gpuarray.zeros(GRID, dtype=np.float32)

max_reduction_gpu(d2, d_partial, np.int32(N2),
                  block=(BLOCK,1,1), grid=(GRID,1,1), shared=BLOCK*4)
cuda.Context.synchronize()

gpu_max = float(d_partial.get().max())   # final reduce on CPU
cpu_max = float(h2.max())

print(f"GPU max = {gpu_max:.6f}")
print(f"CPU max = {cpu_max:.6f}")
print("✓ PASSED" if abs(gpu_max - cpu_max) < 1e-5 else "✗ FAILED")

In [ ]:
# ── Solution Ex 2 ─────────────────────────────────────────────────────────────
max_reduce_solution = """
__global__ void max_reduction(const float *g_idata, float *g_odata, int n) {
    extern __shared__ float sdata[];
    unsigned int tid = threadIdx.x;
    unsigned int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Load: use -infinity as identity for max
    sdata[tid] = (idx < n) ? g_idata[idx] : -1e38f;
    __syncthreads();

    // Tree with fmaxf instead of +
    for (unsigned int s = blockDim.x / 2; s > 0; s >>= 1) {
        if (tid < s) sdata[tid] = fmaxf(sdata[tid], sdata[tid + s]);
        __syncthreads();
    }
    if (tid == 0) g_odata[blockIdx.x] = sdata[0];
}
"""
# Uncomment to check:
# mod_sol2 = SourceModule(max_reduce_solution)
# max_reduction_gpu = mod_sol2.get_function("max_reduction")

---
## Exercise 3 — ReLU Activation  ⭐⭐

**Task:** Apply the ReLU function element-wise:  `out[i] = max(0, in[i])`

This is the core operation in neural network forward passes.  
Each thread handles **one element** — simple but the pattern is used *everywhere* in deep learning.

**Bonus challenge:** After making it work, try doing it with `fmaxf()` in a single line.

```
Input:  [-1.2,  0.5, -0.3,  2.1, -4.0,  0.9]
Output: [ 0.0,  0.5,  0.0,  2.1,  0.0,  0.9]
```

In [ ]:
relu_src = """
__global__ void relu(const float *in, float *out, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n) {
        // TODO: apply ReLU: out[idx] = max(0, in[idx])
        // Hint: use fmaxf(a, b) for float max
        out[idx] = ???;
    }
}
"""

mod_relu = SourceModule(relu_src)
relu_gpu = mod_relu.get_function("relu")

In [ ]:
# ── Test Ex 3 ──────────────────────────────────────────────────────────────────
N3  = 1 << 22
h3  = (np.random.rand(N3).astype(np.float32) - 0.5) * 4   # mix of neg & pos
d3_in  = gpuarray.to_gpu(h3)
d3_out = gpuarray.zeros_like(d3_in)

BLOCK = 256
GRID  = int(np.ceil(N3 / BLOCK))
relu_gpu(d3_in, d3_out, np.int32(N3), block=(BLOCK,1,1), grid=(GRID,1,1))
cuda.Context.synchronize()

expected3 = np.maximum(0, h3)
err3 = np.max(np.abs(d3_out.get() - expected3))
neg_count = int((d3_out.get() < 0).sum())

print(f"Max error              : {err3:.2e}")
print(f"Negative values in out : {neg_count}  (should be 0)")
print("✓ PASSED" if err3 < 1e-6 and neg_count == 0 else "✗ FAILED")

t3 = benchmark_gpu(lambda: relu_gpu(d3_in, d3_out, np.int32(N3), block=(BLOCK,1,1), grid=(GRID,1,1)))
print(f"Time: {t3*1e3:.3f} ms  |  BW: {2*N3*4/t3/1e9:.1f} GB/s")

In [ ]:
# ── Solution Ex 3 ─────────────────────────────────────────────────────────────
relu_solution = """
__global__ void relu(const float *in, float *out, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        out[idx] = fmaxf(0.0f, in[idx]);  // one-liner!
    }
}
"""
# Uncomment to check:
# mod_sol3 = SourceModule(relu_solution)
# relu_gpu = mod_sol3.get_function("relu")

---
## Exercise 4 — Naive Matrix Transpose  ⭐⭐⭐

**Task:** Transpose an M×N matrix: `B[j][i] = A[i][j]`

Use a 2D grid and block. Each thread reads one element from A and writes it to the transposed position in B.

```
A =  [1 2 3]     B = [1 4 7]
     [4 5 6]         [2 5 8]
     [7 8 9]         [3 6 9]
```

**Think:** A thread at `(row, col)` in the output corresponds to `(col, row)` in the input.  
**Warning:** Reads from A are coalesced but writes to B are not (column-stride). We fix this in Ex 5.

In [ ]:
TILE_T = 16

naive_transpose_src = f"""
#define TILE {TILE_T}

__global__ void naive_transpose(const float *A, float *B, int M, int N) {{
    // row/col of the output matrix B
    // TODO: compute row and col using blockIdx and threadIdx (2D)
    int row = ???;  // row index in B  (= col index in A)
    int col = ???;  // col index in B  (= row index in A)

    // TODO: bounds check, then write B[row][col] = A[col][row]
    // Note: A is M×N (row-major), B is N×M (row-major)
    if (??? && ???) {{
        B[??? * M + ???] = A[??? * N + ???];
    }}
}}
"""

mod_t_naive = SourceModule(naive_transpose_src)
naive_transpose = mod_t_naive.get_function("naive_transpose")

In [ ]:
# ── Test Ex 4 ─────────────────────────────────────────────────────────────────
M4, N4 = 1024, 2048
hA4 = np.random.rand(M4, N4).astype(np.float32)
dA4 = gpuarray.to_gpu(hA4)
dB4 = gpuarray.zeros((N4, M4), dtype=np.float32)  # transposed shape

grid4  = (int(np.ceil(N4 / TILE_T)), int(np.ceil(M4 / TILE_T)), 1)
block4 = (TILE_T, TILE_T, 1)
naive_transpose(dA4, dB4, np.int32(M4), np.int32(N4), block=block4, grid=grid4)
cuda.Context.synchronize()

ref4 = hA4.T
err4 = np.max(np.abs(dB4.get() - ref4))
print(f"Max error: {err4:.2e}")
print("✓ PASSED" if err4 < 1e-5 else "✗ FAILED")

t4 = benchmark_gpu(lambda: naive_transpose(dA4, dB4, np.int32(M4), np.int32(N4), block=block4, grid=grid4))
print(f"Time: {t4*1e3:.3f} ms  |  EBW: {2*M4*N4*4/t4/1e9:.1f} GB/s")

In [ ]:
# ── Solution Ex 4 ─────────────────────────────────────────────────────────────
naive_transpose_solution = f"""
#define TILE {TILE_T}

__global__ void naive_transpose(const float *A, float *B, int M, int N) {{
    // Output B is N×M; thread (row,col) computes B[row][col] = A[col][row]
    int row = blockIdx.y * TILE + threadIdx.y;  // row in B = col in A
    int col = blockIdx.x * TILE + threadIdx.x;  // col in B = row in A

    if (row < N && col < M) {{          // bounds in B which is N×M
        B[row * M + col] = A[col * N + row];  // A is M×N row-major
    }}
}}
"""
# Uncomment:
# mod_sol4 = SourceModule(naive_transpose_solution)
# naive_transpose = mod_sol4.get_function("naive_transpose")

---
## Exercise 5 — Tiled Matrix Transpose (Bank-Conflict Free)  ⭐⭐⭐⭐

**Problem with Ex 4:** Writing to B has column-stride — the 32 threads in a warp write to non-contiguous addresses → **uncoalesced writes**.

**Fix:** Load a TILE×TILE block of A into shared memory (coalesced reads), then write it transposed to B (now coalesced writes). Add 1 padding column to `sdata` to avoid shared memory bank conflicts.

```
Step 1: Thread (ty, tx) loads A[row][col] into sdata[ty][tx]   ← coalesced read of A
        __syncthreads()
Step 2: Thread (ty, tx) writes sdata[tx][ty] into B transposed  ← coalesced write to B
```

**Bank conflicts:** Shared memory has 32 banks. Accessing `sdata[0][0], sdata[1][0], ..., sdata[31][0]` (column of TILE×TILE array) hits the same bank repeatedly. Adding 1 to the column dimension (`TILE+1`) staggers the rows across different banks.

In [ ]:
TILE_T = 16

tiled_transpose_src = f"""
#define TILE {TILE_T}

__global__ void tiled_transpose(const float *A, float *B, int M, int N) {{
    // Shared memory with +1 padding to avoid bank conflicts
    __shared__ float sdata[TILE][TILE + 1];

    // Input tile coordinates (in A)
    int x = blockIdx.x * TILE + threadIdx.x;  // col in A
    int y = blockIdx.y * TILE + threadIdx.y;  // row in A

    // TODO: Load tile of A into shared memory (coalesced)
    if (y < M && x < N) {{
        sdata[threadIdx.y][threadIdx.x] = ???;
    }}
    __syncthreads();

    // Output tile coordinates (in B, which is N×M)
    // The block's tile in B is at a swapped position
    int bx = blockIdx.y * TILE + threadIdx.x;  // col in B
    int by = blockIdx.x * TILE + threadIdx.y;  // row in B

    // TODO: Write sdata transposed to B (swapped tx/ty to get coalesced writes)
    if (by < N && bx < M) {{
        B[by * M + bx] = ???;  // Hint: which indices do you swap in sdata?
    }}
}}
"""

mod_t_tiled = SourceModule(tiled_transpose_src)
tiled_transpose = mod_t_tiled.get_function("tiled_transpose")

In [ ]:
# ── Test Ex 5 ─────────────────────────────────────────────────────────────────
M5, N5 = 1024, 2048
hA5 = np.random.rand(M5, N5).astype(np.float32)
dA5 = gpuarray.to_gpu(hA5)
dB5 = gpuarray.zeros((N5, M5), dtype=np.float32)

grid5  = (int(np.ceil(N5 / TILE_T)), int(np.ceil(M5 / TILE_T)), 1)
block5 = (TILE_T, TILE_T, 1)
tiled_transpose(dA5, dB5, np.int32(M5), np.int32(N5), block=block5, grid=grid5)
cuda.Context.synchronize()

err5 = np.max(np.abs(dB5.get() - hA5.T))
print(f"Max error: {err5:.2e}")
print("✓ PASSED" if err5 < 1e-5 else "✗ FAILED")

# Compare naive vs tiled bandwidth
t5 = benchmark_gpu(lambda: tiled_transpose(dA5, dB5, np.int32(M5), np.int32(N5), block=block5, grid=grid5))
t4_ref = benchmark_gpu(lambda: naive_transpose(dA4, dB4, np.int32(M4), np.int32(N4), block=block4, grid=grid4))

bytes_moved = 2 * M5 * N5 * 4
print(f"\nNaive transpose : {t4_ref*1e3:.3f} ms  {bytes_moved/t4_ref/1e9:.1f} GB/s")
print(f"Tiled transpose : {t5*1e3:.3f} ms  {bytes_moved/t5/1e9:.1f} GB/s  [{t4_ref/t5:.2f}× speedup]")

In [ ]:
# ── Solution Ex 5 ─────────────────────────────────────────────────────────────
tiled_transpose_solution = f"""
#define TILE {TILE_T}

__global__ void tiled_transpose(const float *A, float *B, int M, int N) {{
    __shared__ float sdata[TILE][TILE + 1];  // +1 eliminates bank conflicts

    int x = blockIdx.x * TILE + threadIdx.x;
    int y = blockIdx.y * TILE + threadIdx.y;

    // Load: each thread reads one element of A row-major → coalesced
    if (y < M && x < N)
        sdata[threadIdx.y][threadIdx.x] = A[y * N + x];
    __syncthreads();

    // Write: swap block index roles and swap tx/ty in sdata → coalesced
    int bx = blockIdx.y * TILE + threadIdx.x;
    int by = blockIdx.x * TILE + threadIdx.y;
    if (by < N && bx < M)
        B[by * M + bx] = sdata[threadIdx.x][threadIdx.y];  // transpose in smem
}}
"""
# Uncomment:
# mod_sol5 = SourceModule(tiled_transpose_solution)
# tiled_transpose = mod_sol5.get_function("tiled_transpose")

---
## 🏁 Well Done!

| Exercise | Concept practised |
|----------|------------------|
| Ex 1: Vector Scale | Thread index arithmetic, in-place ops |
| Ex 2: Max Reduction | Tree reduction, identity element |
| Ex 3: ReLU | Element-wise conditional, `fmaxf` |
| Ex 4: Naive Transpose | 2D grid/block indexing, uncoalesced writes |
| Ex 5: Tiled Transpose | Shared memory staging, bank conflicts, padding |

### Stretch Challenges
- **Ex 2 extension:** Combine all block maxima on the GPU too (second reduction kernel)
- **Ex 3 extension:** Write a **Leaky ReLU** kernel: `out[i] = in[i] > 0 ? in[i] : 0.01*in[i]`
- **Ex 5 extension:** Measure with Nsight Systems and confirm the bank-conflict disappears
- **New kernel:** Write a CUDA kernel that computes row-wise L2 norms of a matrix